In [8]:
import random
import json
import math
from faker import Faker

# Install faker if not available
try:
    from faker import Faker
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'faker'], check=True)
    from faker import Faker

fake = Faker()
print('✅ Dependencies ready.')

✅ Dependencies ready.


In [ ]:
# ── USER INPUTS ──────────────────────────────────────────────
NUM_COURSES  = 5
NUM_STUDENTS = 20
NUM_FACULTY  = 10

# Guard: need at least 2 faculty per course (1 teacher + 1 scrutinizer min)
if NUM_FACULTY < 2:
    raise ValueError('At least 2 faculty members are required.')

print(f'\n📋 Generating data for {NUM_COURSES} courses, '
      f'{NUM_STUDENTS} students, {NUM_FACULTY} faculties...')


📋 Generating data for 5 courses, 20 students, 10 faculties...


## 🏗️ Data Generation

In [4]:
# ── HELPER: generate scrutinizer mark close to teacher mark ──
def scrutinized_mark(teacher_mark: float, std_dev: float = 5.0) -> float:
    """Return a mark near teacher_mark, clamped to [0, 100]."""
    delta = random.gauss(0, std_dev)
    return round(max(0.0, min(100.0, teacher_mark + delta)), 2)


# ── 1. FACULTIES ─────────────────────────────────────────────
faculties = [
    {
        'faculty_id': f'FAC{str(i+1).zfill(3)}',
        'name': fake.name()
    }
    for i in range(NUM_FACULTY)
]


# ── 2. STUDENTS ──────────────────────────────────────────────
students = [
    {
        'student_id': f'STU{str(i+1).zfill(4)}',
        'name': fake.name()
    }
    for i in range(NUM_STUDENTS)
]


# ── 3. COURSES + ENROLLMENTS + MARKS ─────────────────────────
all_faculty_ids  = [f['faculty_id'] for f in faculties]
all_student_ids  = [s['student_id'] for s in students]

courses = []

for i in range(NUM_COURSES):
    course_id   = f'CSE{str(i+1).zfill(3)}'
    course_name = f'CSE{str(i+1).zfill(3)}'

    # -- How many teachers & scrutinizers for this course? ----
    max_roles     = min(3, math.floor(NUM_FACULTY / 2))   # 1–3, but bounded
    num_teachers  = random.randint(1, max(1, max_roles))
    num_scrutinizers = random.randint(1, max(1, max_roles))

    # -- Pick teachers first, then scrutinizers from the rest -
    shuffled = random.sample(all_faculty_ids, len(all_faculty_ids))
    teachers     = shuffled[:num_teachers]
    remaining    = shuffled[num_teachers:]
    # Clamp scrutinizers to what's left
    num_scrutinizers = min(num_scrutinizers, len(remaining))
    scrutinizers = remaining[:num_scrutinizers]

    # -- Random enrollment (10 % – 80 % of total students) ----
    min_enroll = max(1, math.ceil(NUM_STUDENTS * 0.10))
    max_enroll = max(min_enroll, math.floor(NUM_STUDENTS * 0.80))
    num_enrolled = random.randint(min_enroll, max_enroll)
    enrolled_ids = random.sample(all_student_ids, num_enrolled)

    # -- Generate marks for each enrolled student --------------
    enrollments = []
    for sid in enrolled_ids:
        # Each teacher gives a mark independently
        teacher_marks = {
            tid: round(random.uniform(0, 100), 2)
            for tid in teachers
        }
        # Scrutinizer corrects based on average teacher mark
        avg_teacher_mark = sum(teacher_marks.values()) / len(teacher_marks)
        scrutinizer_marks = {
            scr: scrutinized_mark(avg_teacher_mark)
            for scr in scrutinizers
        }

        enrollments.append({
            'student_id'       : sid,
            'teacher_marks'    : teacher_marks,      # {faculty_id: mark}
            'scrutinizer_marks': scrutinizer_marks   # {faculty_id: corrected_mark}
        })

    courses.append({
        'course_id'   : course_id,
        'course_name' : course_name,
        'teachers'    : teachers,
        'scrutinizers': scrutinizers,
        'enrollments' : enrollments
    })


# ── 4. ASSEMBLE TOP-LEVEL JSON ────────────────────────────────
mock_data = {
    'metadata': {
        'num_courses' : NUM_COURSES,
        'num_students': NUM_STUDENTS,
        'num_faculty' : NUM_FACULTY
    },
    'faculties': faculties,
    'students' : students,
    'courses'  : courses
}

print('✅ Data generated successfully!')
print(f'   Courses  : {len(courses)}')
print(f'   Students : {len(students)}')
print(f'   Faculties: {len(faculties)}')

✅ Data generated successfully!
   Courses  : 5
   Students : 20
   Faculties: 10


Preview (first course)

In [5]:
# Pretty-print first course as a sanity check
preview = {
    'metadata' : mock_data['metadata'],
    'faculties' : mock_data['faculties'][:3],   # first 3
    'students'  : mock_data['students'][:3],    # first 3
    'courses'   : [
        {
            **mock_data['courses'][0],
            'enrollments': mock_data['courses'][0]['enrollments'][:2]  # first 2 enrollments
        }
    ]
}
print(json.dumps(preview, indent=2))

{
  "metadata": {
    "num_courses": 5,
    "num_students": 20,
    "num_faculty": 10
  },
  "faculties": [
    {
      "faculty_id": "FAC001",
      "name": "Steven Miles"
    },
    {
      "faculty_id": "FAC002",
      "name": "Kristen Valenzuela"
    },
    {
      "faculty_id": "FAC003",
      "name": "Donna Davis"
    }
  ],
  "students": [
    {
      "student_id": "STU0001",
      "name": "Amanda Johnson"
    },
    {
      "student_id": "STU0002",
      "name": "Christopher Fernandez"
    },
    {
      "student_id": "STU0003",
      "name": "Sarah Rogers"
    }
  ],
  "courses": [
    {
      "course_id": "CSE001",
      "course_name": "CSE001",
      "teachers": [
        "FAC004",
        "FAC001",
        "FAC005"
      ],
      "scrutinizers": [
        "FAC008",
        "FAC007",
        "FAC002"
      ],
      "enrollments": [
        {
          "student_id": "STU0009",
          "teacher_marks": {
            "FAC004": 19.78,
            "FAC001": 79.23,
            "

In [6]:
OUTPUT_FILE = 'academic_mock_data.json'

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(mock_data, f, indent=2, ensure_ascii=False)

import os
size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f'✅ Saved → {OUTPUT_FILE}  ({size_kb:.1f} KB)')

✅ Saved → academic_mock_data.json  (20.9 KB)


## 📊 Quick Stats

In [7]:
total_enrollments = sum(len(c['enrollments']) for c in courses)
avg_enrolled = total_enrollments / NUM_COURSES if NUM_COURSES else 0

print('=' * 45)
print('          DATASET SUMMARY')
print('=' * 45)
print(f'  Total courses       : {NUM_COURSES}')
print(f'  Total students      : {NUM_STUDENTS}')
print(f'  Total faculties     : {NUM_FACULTY}')
print(f'  Total enrollments   : {total_enrollments}')
print(f'  Avg students/course : {avg_enrolled:.1f}')
print('=' * 45)

print('\nPer-course breakdown:')
print(f'{"Course":<10} {"Teachers":<10} {"Scrutinizers":<14} {"Students"}')
print('-' * 45)
for c in courses:
    print(f"{c['course_id']:<10} {len(c['teachers']):<10} "
          f"{len(c['scrutinizers']):<14} {len(c['enrollments'])}")

          DATASET SUMMARY
  Total courses       : 5
  Total students      : 20
  Total faculties     : 10
  Total enrollments   : 65
  Avg students/course : 13.0

Per-course breakdown:
Course     Teachers   Scrutinizers   Students
---------------------------------------------
CSE001     3          3              15
CSE002     2          1              12
CSE003     2          1              9
CSE004     1          3              14
CSE005     2          2              15
